# Study 921 — Bill Ladder vs ETF — the teardown

The ladder construction and its one execution lag, the discount→bond-equivalent conversion, the total-return race against three cash funds, the Newey-West *t* on the daily difference plus the full **inference audit** (the bounce diagnostic, the disclosed HAC-bandwidth and bootstrap-block scans, and the knob-free non-overlapping test that actually carries the verdict), the gross-of-fee attribution, the era and rate-level cuts, two friction sweeps, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `1e3d76fa1bfa`).

In [1]:
R = {'start': '2007-05-31', 'end': '2026-06-30', 'n_days': 4799, 'n_rolls': 992, 'fp': '1e3d76fa1bfa', 'bil_cagr_l': 1.4928, 'bil_cagr_e': 1.3616, 'bil_gap': 12.83, 'bil_t': 2.75, 'bil_tnaive': 1.15, 'sgov_start': '2020-06-02', 'sgov_n': 1527, 'sgov_cagr_l': 2.9855, 'sgov_cagr_e': 2.9571, 'sgov_gap': 2.76, 'sgov_t': 0.62, 'shv_n': 4894, 'shv_cagr_l': 1.5639, 'shv_cagr_e': 1.5801, 'shv_gap': -1.65, 'shv_t': -0.34, 'vol_ladder': 0.1464, 'vol_bil': 0.4885, 'bil_er': 13.54, 'bil_gross': 149.7, 'bil_resid': -0.4, 'sgov_er': 9.0, 'sgov_gross': 304.7, 'sgov_resid': -6.2, 'shv_er': 15.0, 'shv_gross': 173.0, 'shv_resid': -16.6, 'ci_lo': 5.32, 'ci_hi': 20.46, 'ci_neg': 0.05, 'era1_n': 2163, 'era1_rate': 0.49, 'era1_gap': 10.46, 'era1_t': 1.03, 'era2_n': 1510, 'era2_rate': 0.93, 'era2_gap': 14.17, 'era2_t': 3.23, 'era3_n': 1126, 'era3_rate': 3.97, 'era3_gap': 15.61, 'era3_t': 2.04, 'zr_n': 2785, 'zr_rate': 0.15, 'zr_gap': 12.64, 'zr_t': 1.94, 'nr_n': 2014, 'nr_rate': 3.23, 'nr_gap': 13.08, 'nr_t': 1.68, 'raw_gap': 9.44, 'raw_t': 2.02, 'rung4_gap': 12.79, 'rung26_gap': 12.82, 'c1_gap': 8.83, 'c1_t': 1.89, 'c2_gap': 4.83, 'c2_t': 1.03, 'c3_gap': 0.83, 'c3_t': 0.18, 'c5_gap': -7.17, 'c5_t': -1.53, 'c10_gap': -27.16, 'c10_t': -5.81, 'i1_gap': 11.21, 'i1_t': 2.4, 'i2_gap': 9.59, 'i2_t': 2.05, 'i3_gap': 7.98, 'i3_t': 1.71, 'i5_gap': 4.74, 'i5_t': 1.01, 'conv_high': 8.7, 'conv_low': 1.1, 'syn_planted': 13.35, 'syn_planted_sd': 0.53, 'syn_null': -0.16, 'syn_null_sd': 0.53, 'syn_fire': 0, 'acf1': -0.366, 'hac_scan': ((0, 1.15), (1, 1.44), (2, 1.73), (5, 2.21), (9, 2.75), (21, 3.45), (63, 5.54), (252, 9.26)), 'boot_blocks': ((5, 1.15, 24.07, 1.8), (10, 3.74, 22.19, 0.1), (21, 5.32, 20.46, 0.05), (63, 8.37, 17.25, 0.0)), 'nov': (('weekly', 997, 0.246, 2.18), ('monthly', 230, 1.064, 3.27), ('quarterly', 77, 3.179, 3.54)), 'nov_raw': (('weekly', 1.6), ('monthly', 2.41), ('quarterly', 2.6)), 'era1_tm': 1.32, 'era2_tm': 4.02, 'era3_tm': 2.96}

## Construction

13 rungs of a 91-day bill, one bought every 7 calendar days, each held to maturity. The ladder's daily accrual is the equal-weighted mean of the 13 live rungs' locked bond-equivalent yields, applied actual/365 over **calendar** days — so a Friday→Monday step pays three days, the same clock the ETF's total-return close runs on.

**The lag is exactly one.** The ^IRX quote at the close of day *t* prices the bill bought at *t+1*. Nothing else looks forward.

**The conversion is not cosmetic.** ^IRX is a bank-discount rate, actual/360:

```
P   = 1 - d * 91 / 360
BEY = (1 - P) / P * 365 / 91
```

worth **+8.7 bps** at a 3.70% quote and **+1.1 bps** at 0.50% — the same order as the effect under test, which is why the raw-quote variant is reported as a floor.

> 💡 **In plain words** — a discount rate is quoted off the face value you get back, not off the smaller price you paid, and on a 360-day year. Both corrections push the real yield up. Skip them and you understate the ladder by most of a fee.

## The headline race — total return vs total return

Cash is the numeraire; there is nothing to take excess of. The statistic is the annualised mean daily return difference in bps/yr.

In [2]:
print(f"BIL  n={R['n_days']:5d}  ladder {R['bil_cagr_l']:.4f}%  ETF {R['bil_cagr_e']:.4f}%  "
      f"gap {R['bil_gap']:+6.2f} bps/yr  HAC t {R['bil_t']:+.2f}  (naive t {R['bil_tnaive']:+.2f})")
print(f"SGOV n={R['sgov_n']:5d}  ladder {R['sgov_cagr_l']:.4f}%  ETF {R['sgov_cagr_e']:.4f}%  "
      f"gap {R['sgov_gap']:+6.2f} bps/yr  HAC t {R['sgov_t']:+.2f}")
print(f"SHV  n={R['shv_n']:5d}  ladder {R['shv_cagr_l']:.4f}%  ETF {R['shv_cagr_e']:.4f}%  "
      f"gap {R['shv_gap']:+6.2f} bps/yr  HAC t {R['shv_t']:+.2f}   <- duration control")
print(f"\nvol: ladder {R['vol_ladder']:.4f}% vs BIL {R['vol_bil']:.4f}% "
      f"-- amortised cost, NOT less risk; no Sharpe race is quoted")

BIL  n= 4799  ladder 1.4928%  ETF 1.3616%  gap +12.83 bps/yr  HAC t +2.75  (naive t +1.15)
SGOV n= 1527  ladder 2.9855%  ETF 2.9571%  gap  +2.76 bps/yr  HAC t +0.62
SHV  n= 4894  ladder 1.5639%  ETF 1.5801%  gap  -1.65 bps/yr  HAC t -0.34   <- duration control

vol: ladder 0.1464% vs BIL 0.4885% -- amortised cost, NOT less risk; no Sharpe race is quoted


## The inference audit — because HAC *helps* us here

Naive *t* = +1.15, HAC *t* = +2.75. HAC usually *deflates* a *t*; here it more than doubles it. A correction that moves the result the author's way has to be audited, not asserted, so this section answers three questions in order: **why** the naive *t* is wrong, **how much** the tuning knobs move things, and **what the answer is with no knob at all**.

**1. Why.** A bill ETF's daily close carries bid-offer bounce (Roll 1984), so the ladder-minus-ETF difference is a first difference of a stationary pricing error plus a drift, hence **negatively** autocorrelated at lag 1. Negative autocovariance shrinks the variance of the sample mean, so the i.i.d. standard error is too *large*.

In [3]:
print(f"lag-1 autocorrelation of the daily difference: {R['acf1']:+.3f}")
print('strongly negative = the Roll (1984) bounce signature. The naive SE is too big.')

lag-1 autocorrelation of the daily difference: -0.366
strongly negative = the Roll (1984) bounce signature. The naive SE is too big.


**2. The knobs, disclosed.** HAC has a bandwidth; the block bootstrap has a block length. On this tape *both* push the same way — more lags, more significance — so the honest thing is to show the whole range and note where the headline sits in it.

In [4]:
print('HAC t vs bandwidth:')
for lags, t in R['hac_scan']:
    tag = 'naive (i.i.d.)' if lags == 0 else f'HAC {lags:3d} lags'
    mark = '   <- automatic rule, the headline' if lags == 9 else ''
    print(f"  {tag:16s} t {t:+.2f}{mark}")
print('\nbootstrap CI vs block length:')
for b, lo, hi, neg in R['boot_blocks']:
    print(f"  block {b:3d}d  95% CI [{lo:+6.2f}, {hi:+6.2f}]  share<0 {neg:.2f}%")
print('\nThe automatic bandwidth sits near the BOTTOM of the kernel family, and the')
print('shortest block gives the widest CI -- the headline is the conservative pick,')
print('not the flattering one. But neither of these settles the question on its own.')

HAC t vs bandwidth:
  naive (i.i.d.)   t +1.15
  HAC   1 lags     t +1.44
  HAC   2 lags     t +1.73
  HAC   5 lags     t +2.21
  HAC   9 lags     t +2.75   <- automatic rule, the headline
  HAC  21 lags     t +3.45
  HAC  63 lags     t +5.54
  HAC 252 lags     t +9.26

bootstrap CI vs block length:
  block   5d  95% CI [ +1.15, +24.07]  share<0 1.80%
  block  10d  95% CI [ +3.74, +22.19]  share<0 0.10%
  block  21d  95% CI [ +5.32, +20.46]  share<0 0.05%
  block  63d  95% CI [ +8.37, +17.25]  share<0 0.00%

The automatic bandwidth sits near the BOTTOM of the kernel family, and the
shortest block gives the widest CI -- the headline is the conservative pick,
not the flattering one. But neither of these settles the question on its own.


**3. The arbiter, which has no knob at all.** Sum the daily difference into **non-overlapping** calendar periods. Inside each period the bounce telescopes away (only the endpoints survive) while the accrual gap accumulates, so consecutive period sums are close to independent and an *ordinary* one-sample *t* is valid as it stands. There is no bandwidth and no block length to choose.

This is the test the Real stamp actually rests on. HAC and the bootstrap merely concur.

In [5]:
print('non-overlapping period sums -- ordinary t, nothing to tune:')
for lab, n, mean, t in R['nov']:
    print(f"  {lab:10s} n={n:4d} periods  mean {mean:+.3f} bps/period  t {t:+.2f}")
print(f"\n  ... vs the naive daily t of {R['bil_tnaive']:+.2f} on the same data.")
print('  The knob-free test agrees with HAC, not with naive. Verdict evidence.')
print('\nsame test on the CONSERVATIVE raw-quote convention (no discount->BEY):')
for lab, t in R['nov_raw']:
    flag = '' if t >= 2.0 else '   <- honestly short of 2'
    print(f"  {lab:10s} t {t:+.2f}{flag}")

non-overlapping period sums -- ordinary t, nothing to tune:
  weekly     n= 997 periods  mean +0.246 bps/period  t +2.18
  monthly    n= 230 periods  mean +1.064 bps/period  t +3.27
  quarterly  n=  77 periods  mean +3.179 bps/period  t +3.54

  ... vs the naive daily t of +1.15 on the same data.
  The knob-free test agrees with HAC, not with naive. Verdict evidence.

same test on the CONSERVATIVE raw-quote convention (no discount->BEY):
  weekly     t +1.60   <- honestly short of 2
  monthly    t +2.41
  quarterly  t +2.60


> ⚠️ **What this does not rescue.** The raw-quote floor clears at monthly and above but falls to +1.60 weekly, and the ladder leg is still a *simulation* priced off a secondary-market quote. The inference is sound; the instrument is still modelled.

## Attribution — the gap *is* the expense ratio

Add each fund's published expense ratio back to its net return to recover its gross return, and read the residual against the ladder. Expense ratios are a **PROXY** (sponsor stickers, not tape) and never enter a return calculation.

In [6]:
for tag, cl, er, gr, res in [
    ('BIL ', R['bil_cagr_l']*100, R['bil_er'], R['bil_gross'], R['bil_resid']),
    ('SGOV', R['sgov_cagr_l']*100, R['sgov_er'], R['sgov_gross'], R['sgov_resid']),
    ('SHV ', R['shv_cagr_l']*100, R['shv_er'], R['shv_gross'], R['shv_resid'])]:
    print(f"{tag}: ladder {cl:7.1f} bps/yr  vs ETF gross {gr:7.1f} (ER {er:5.2f})  "
          f"-> residual {res:+6.1f} bps/yr")
print('\nBIL residual -0.4 bps over 19 years: no curve pickup, no tenor bonus, no skill.')
print('SGOV -6.2: its EFFECTIVE whole-period fee was ~3 bps, not the 9 bps sticker.')
print('SHV -16.6: the duration control earning a curve pickup the ladder forgoes.')

BIL : ladder   149.3 bps/yr  vs ETF gross   149.7 (ER 13.54)  -> residual   -0.4 bps/yr
SGOV: ladder   298.6 bps/yr  vs ETF gross   304.7 (ER  9.00)  -> residual   -6.2 bps/yr
SHV : ladder   156.4 bps/yr  vs ETF gross   173.0 (ER 15.00)  -> residual  -16.6 bps/yr

BIL residual -0.4 bps over 19 years: no curve pickup, no tenor bonus, no skill.
SGOV -6.2: its EFFECTIVE whole-period fee was ~3 bps, not the 9 bps sticker.
SHV -16.6: the duration control earning a curve pickup the ladder forgoes.


> 💡 **In plain words** — if the ladder were doing anything other than dodging a fee, this residual would not be zero. It is zero for BIL, negative for the longer-maturity fund (which earns something the ladder can't), and negative for SGOV in the amount by which its sticker overstates what it actually charged.

## Era cut and rate-level cut

The date slabs are cut out of the already-built difference series, so no slab loses a quarter to ladder warmup and the three partition the sample exactly. The rate-level cut is the sharper test: a fee is level-invariant, a carry is not.

In [7]:
print('era                     gap      HAC t   knob-free monthly t')
for lab, n, rate, gap, t, tm in [
    ('2007-2015', R['era1_n'], R['era1_rate'], R['era1_gap'], R['era1_t'], R['era1_tm']),
    ('2016-2021', R['era2_n'], R['era2_rate'], R['era2_gap'], R['era2_t'], R['era2_tm']),
    ('2022-2026', R['era3_n'], R['era3_rate'], R['era3_gap'], R['era3_t'], R['era3_tm'])]:
    flag = '' if tm >= 2.0 else '   <- sign only, NOT significant'
    print(f"{lab} n={n:5d} quote {rate:.2f}%  {gap:+6.2f}   {t:+.2f}    {tm:+.2f}{flag}")
print('\n-> the SIGN is positive in all three, but only eras 2 and 3 are individually')
print('   significant. "Positive in all three eras" is not three endorsements.')
print()
print(f"quote <1%  n={R['zr_n']:5d}  mean {R['zr_rate']:.2f}%  "
      f"gap {R['zr_gap']:+6.2f} (t={R['zr_t']:+.2f})")
print(f"quote >=1% n={R['nr_n']:5d}  mean {R['nr_rate']:.2f}%  "
      f"gap {R['nr_gap']:+6.2f} (t={R['nr_t']:+.2f})")
print('\n-> a 22x change in the rate level moves the gap by 0.44 bps. That is a fee.')

era                     gap      HAC t   knob-free monthly t
2007-2015 n= 2163 quote 0.49%  +10.46   +1.03    +1.32   <- sign only, NOT significant
2016-2021 n= 1510 quote 0.93%  +14.17   +3.23    +4.02
2022-2026 n= 1126 quote 3.97%  +15.61   +2.04    +2.96

-> the SIGN is positive in all three, but only eras 2 and 3 are individually
   significant. "Positive in all three eras" is not three endorsements.

quote <1%  n= 2785  mean 0.15%  gap +12.64 (t=+1.94)
quote >=1% n= 2014  mean 3.23%  gap +13.08 (t=+1.68)

-> a 22x change in the rate level moves the gap by 0.44 bps. That is a fee.


## Assumption and construction sweeps

In [8]:
print(f"discount->BEY (headline) : gap {R['bil_gap']:+6.2f} (t={R['bil_t']:+.2f})")
print(f"raw quote (conservative) : gap {R['raw_gap']:+6.2f} (t={R['raw_t']:+.2f})  "
      f"<- survives, but only just")
print(f" 4 rungs (monthly roll)  : gap {R['rung4_gap']:+6.2f}")
print(f"26 rungs (2x weekly)     : gap {R['rung26_gap']:+6.2f}  "
      f"<- schedule moves it by 0.04 bps")

discount->BEY (headline) : gap +12.83 (t=+2.75)
raw quote (conservative) : gap  +9.44 (t=+2.02)  <- survives, but only just
 4 rungs (monthly roll)  : gap +12.79
26 rungs (2x weekly)     : gap +12.82  <- schedule moves it by 0.04 bps


## Friction sweeps — where it breaks

52 rolls a year on 1/13 of NAV, so per-auction friction costs about **4x** its per-roll size in annual drag. Both frictions are PROXY / ASSUMPTION and swept rather than assumed. No short leg anywhere, so no borrow.

In [9]:
for c, g, t in [(0, R['bil_gap'], R['bil_t']), (1, R['c1_gap'], R['c1_t']),
                (2, R['c2_gap'], R['c2_t']), (3, R['c3_gap'], R['c3_t']),
                (5, R['c5_gap'], R['c5_t']), (10, R['c10_gap'], R['c10_t'])]:
    flag = '  <- edge gone' if (c > 0 and abs(t) < 2) else ('  <- now a LOSS' if g < 0 else '')
    print(f"{c:2d} bps/auction : gap {g:+7.2f} (t={t:+.2f}){flag}")
print()
for d, g, t in [(0, R['bil_gap'], R['bil_t']), (1, R['i1_gap'], R['i1_t']),
                (2, R['i2_gap'], R['i2_t']), (3, R['i3_gap'], R['i3_t']),
                (5, R['i5_gap'], R['i5_t'])]:
    print(f"{d:2d} idle days   : gap {g:+7.2f} (t={t:+.2f})")

 0 bps/auction : gap  +12.83 (t=+2.75)
 1 bps/auction : gap   +8.83 (t=+1.89)  <- edge gone
 2 bps/auction : gap   +4.83 (t=+1.03)  <- edge gone
 3 bps/auction : gap   +0.83 (t=+0.18)  <- edge gone
 5 bps/auction : gap   -7.17 (t=-1.53)  <- edge gone
10 bps/auction : gap  -27.16 (t=-5.81)  <- now a LOSS

 0 idle days   : gap  +12.83 (t=+2.75)
 1 idle days   : gap  +11.21 (t=+2.40)
 2 idle days   : gap   +9.59 (t=+2.05)
 3 idle days   : gap   +7.98 (t=+1.71)
 5 idle days   : gap   +4.74 (t=+1.01)


## Live synthetic control — the machinery is unbiased

An Ornstein-Uhlenbeck short rate, the matching 13-week discount quote, and a cash ETF that accrues that rate minus a *known* fee plus bid-offer bounce. The pipeline must recover the planted fee and must find nothing when the fee is zero. This proves the harness neither invents nor eats basis points; it never supports the real-tape stamp.

In [10]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from bill_ladder import data, strategy as st
pl = np.array([st.synthetic_detect(
    data.synthetic_daily(signal_strength=1.0, seed=921+s)[0])['gap_bps'] for s in range(8)])
nl = np.array([st.synthetic_detect(
    data.synthetic_daily(signal_strength=0.0, seed=921+s)[0])['gap_bps'] for s in range(8)])
print(f"planted 13.50 bps fee x8: recovered {pl.mean():+.2f} (sd {pl.std(ddof=1):.2f})")
print(f"free-ETF null        x8: recovered {nl.mean():+.2f} (sd {nl.std(ddof=1):.2f}), "
      f"|gap|>=4 bps in {(np.abs(nl)>=4).sum()}/8")
half, tr = data.synthetic_daily(signal_strength=0.5, seed=921)
print(f"half fee ({tr['fee_bps_effective']:.2f} bps)  : recovered "
      f"{st.synthetic_detect(half)['gap_bps']:+.2f} -- the response is linear")

planted 13.50 bps fee x8: recovered +13.35 (sd 0.53)
free-ETF null        x8: recovered -0.16 (sd 0.53), |gap|>=4 bps in 0/8


half fee (6.75 bps)  : recovered +6.92 -- the response is linear


## Verdict

- **Signal — Real.** Gap **+12.83 bps/yr** vs BIL. The significance is carried by the test with **no tuning knob** — non-overlapping period sums at *t* = **+2.18 / +3.27 / +3.54** (weekly / monthly / quarterly) — with HAC (+2.75) and the block bootstrap (**[+5.32, +20.46]**, 0.05% of draws negative) concurring, and the naive daily *t* of +1.15 understood to be too small because the difference has a lag-1 autocorrelation of -0.37. Flat in the level of rates (+12.64 at 0.15% vs +13.08 at 3.23%), invariant to the rung schedule, and surviving the conservative raw-quote convention (+9.44, monthly *t* = +2.41). The gross-of-fee residual against BIL is **-0.4 bps/yr**: the gap is the expense ratio, fully attributed. The synthetic control recovers a planted fee (+13.35, sd 0.53) and is silent on the null (-0.16, 0/8). **Caveats named, not buried:** the ladder leg is *simulated* from a rate index rather than traded and ^IRX is a secondary-market quote, not the auction stop-out; era 1 (45% of the sample) is positive but **not individually significant** (monthly *t* = +1.32); and the raw-quote floor clears at monthly and above but not at weekly (+1.60). An accounting identity confirmed, not a discovery.
- **Tradability — Fragile.** The headline charges **no friction to either leg**. The edge is a fee, so it is capped by the fee — and it dies at **3 bps** of per-auction friction (+0.83, *t* = +0.18) or **3 idle days** per roll (+7.98, *t* = +1.71). Against SGOV it is already only +2.76 bps (*t* = +0.62), so the one-click substitute captures most of it. And the ladder's 0.15% volatility against BIL's 0.49% is amortised-cost accounting, not risk reduction — the ladder is the same credit and the same tenor, held less liquidly.
- **Survivorship.** No cross-section, so no classic survivorship bias; but the three funds raced are three that survived, and the expensive cash ETFs that closed are absent. That omission biases *toward* the ladder, not against it.